In [ ]:
# ch04_CUSTOMIZED_STATE_AND_TIME_TRAVEL.ipynb

In [32]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages
from langchain.chat_models import init_chat_model
from langchain_tavily import TavilySearch
from langgraph.graph import StateGraph, START
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
import os

from dotenv import load_dotenv
load_dotenv()

#llm = init_chat_model("openai:gpt-4.1")
# llm = init_chat_model(
#     model=f"google_genai:{os.getenv('GEMINI_MODEL')}",
#     api_key=os.getenv('GEMINI_API_KEY'),
# )

llm = init_chat_model(
    model=f"bedrock:{os.getenv('BEDROCK_MODEL')}",
    region_name=os.getenv('BEDROCK_REGION', 'ap-southeast-2'),
)

class State(TypedDict):
    messages: Annotated[list, add_messages]
    name: str
    release_date: str

ValidationError: 1 validation error for ChatBedrock
  Value error, Could not load credentials to authenticate with AWS client. Please check that the specified profile name and/or its credentials are valid. Service error: You must specify a region. [type=value_error, input_value={'model': 'apac.anthropic...se_converse_api': False}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error

In [17]:
from langchain_core.messages import ToolMessage
from langchain_core.tools import InjectedToolCallId, tool
from langgraph.types import Command, interrupt

@tool
# 상태 업데이트를 위한 ToolMessage를 생성하는 경우,
# 일반적으로 해당 도구 호출(tool call)에 대한 ID가 필요합니다.
# 이때 LangChain의 InjectedToolCallId를 사용하면,
# 해당 인자가 도구의 스키마(schema)에 모델에게 노출되지 않도록 처리할 수 있습니다.
def human_assistance(
    name: str, release_date: str, tool_call_id: Annotated[str, InjectedToolCallId]
):
    """사람의 확인이 필요한 정보를 검토받는 도구입니다."""

    # 사람에게 정보가 맞는지 물어봅니다.
    human_response = interrupt(
        {
            "question": "이 정보가 맞나요?",
            "name": name,
            "release_date": release_date,
        }
    )

    # 사람이 "Yes"로 응답한 경우 상태를 그대로 사용
    if human_response.get("correct", "").lower().startswith("y"):
        verified_name = name
        verified_date = release_date
        response = "정보가 정확하다고 확인됨."
    # 수정이 필요한 경우, 사람이 제공한 값을 사용
    else:
        verified_name = human_response.get("name", name)
        verified_date = human_response.get("release_date", release_date)
        response = f"사람이 수정한 정보: {human_response}"

    # 상태에 name, release_date 저장하고, 메시지도 함께 반환
    state_update = {
        "name": verified_name,
        "release_date": verified_date,
        "messages": [ToolMessage(response, tool_call_id=tool_call_id)],
    }

    # 상태 갱신을 위해 Command 객체를 반환
    return Command(update=state_update)

In [18]:
search_tool = TavilySearch(max_results=2)
tools = [search_tool, human_assistance]
llm_with_tools = llm.bind_tools(tools)

def chatbot(state: State):
    message = llm_with_tools.invoke(state["messages"])
    assert(len(message.tool_calls) <= 1)
    return {"messages": [message]}

graph_builder = StateGraph(State)
graph_builder.add_node("chatbot", chatbot)
tool_node = ToolNode(tools=tools)
graph_builder.add_node("tools", tool_node)

graph_builder.add_conditional_edges("chatbot", tools_condition)
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge(START, "chatbot")

memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)

In [19]:
user_input = (
    "랭그래프가 언제 출시되었는지 찾아줄래요? "
    "결과를 찾으면 human_assistance 도구를 사용해서 사람에게 확인해줘."
)

config = {"configurable": {"thread_id": "1"}}

events = graph.stream(
    {"messages": [{"role": "user", "content": user_input}]},
    config,
    stream_mode="values",
)

for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

================================ Human Message =================================

랭그래프가 언제 출시되었는지 찾아줄래요? 결과를 찾으면 human_assistance 도구를 사용해서 사람에게 확인해줘.
================================== Ai Message ==================================
Tool Calls:
  tavily_search (8ed59060-4a80-4528-93a7-6aa6eee8ecc4)
 Call ID: 8ed59060-4a80-4528-93a7-6aa6eee8ecc4
  Args:
    query: 랭그래프 출시일
================================= Tool Message =================================
Name: tavily_search

{"query": "랭그래프 출시일", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.langchain.com/langgraph", "title": "LangGraph: Agent Orchestration Framework for Reliable AI Agents", "content": "[![Image 7](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/6989c02453d869396317aaa3_updated-1.svg) deepagents Build long-running agents for complex tasks](https://www.langchain.com/deep-agents)[![Image 8](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/6989c024409fcfc7e5b

In [12]:
snapshot = graph.get_state(config)
print(snapshot.next)  # 출력: ('tools',)

()


In [20]:
from langgraph.types import Command

human_command = Command(
    resume={
        "name": "랭그래프",
        "release_date": "2024년 1월 17일",
    },
)

events = graph.stream(human_command, config, stream_mode="values")

for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

================================== Ai Message ==================================
Tool Calls:
  human_assistance (4f9b2554-2f70-4c0e-b4cf-ed35f2b49421)
 Call ID: 4f9b2554-2f70-4c0e-b4cf-ed35f2b49421
  Args:
    name: 랭그래프
    release_date: 2025-01-09
================================= Tool Message =================================
Name: human_assistance

사람이 수정한 정보: {'name': '랭그래프', 'release_date': '2024년 1월 17일'}
================================== Ai Message ==================================

[{'type': 'text', 'text': '랭그래프는 2024년 1월 17일에 출시되었습니다.', 'extras': {'signature': 'CsEBAQw51sdJXdYCVkBX9bN5LIUm5fLdRj4neH7WNcH0unQRBWqB8QgBhj843rqSQoZzDPSQtYZkP2r6Vb16mH9YQBICj7TqSWAw44duRsSCHUVLTrINtlGnOydrxevuB/U+mJsFy/PwiNF5WHWz6UJav4Dq6dBpNej/962ZG+dkYdf3QnDQk4cqVaqnUYOqBLU48egaYcgph7j1RlzKYydPRk7duFm2WGDpI/OCV3HF0NN893W5w1vdAzGLYKidXZvyiQ=='}}]


In [21]:
snapshot = graph.get_state(config)
{k: v for k, v in snapshot.values.items() if k in ("name", "release_date")}

{'name': '랭그래프', 'release_date': '2024년 1월 17일'}

In [22]:
graph.update_state(config, {"name": "LangGraph (library)"})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f163013-7f52-68b2-8006-3e90883eb5da'}}

In [23]:
snapshot = graph.get_state(config)
{k: v for k, v in snapshot.values.items() if k in ("name", "release_date")}

{'name': 'LangGraph (library)', 'release_date': '2024년 1월 17일'}

In [11]:
# TIME_TRAVEL

In [28]:
# 그래프의 상태 기록 전체를 가져옵니다 (지금까지의 실행 히스토리)
history = graph.get_state_history(config)
to_replay = None
for state in history:
    # 각 상태에서 메시지 개수와 다음 노드를 출력합니다
    print("Messages:", len(state.values["messages"]), "Next:", state.next)
    print("-" * 80)
    
    # 메시지가 6개인 상태를 선택합니다 (임의 기준으로 선택)
    if len(state.values["messages"]) == 3:
        # 나중에 되감기(replay)할 상태로 저장합니다
        to_replay = state

print(to_replay)

Messages: 6 Next: ()
--------------------------------------------------------------------------------
Messages: 6 Next: ()
--------------------------------------------------------------------------------
Messages: 5 Next: ('chatbot',)
--------------------------------------------------------------------------------
Messages: 4 Next: ('tools',)
--------------------------------------------------------------------------------
Messages: 3 Next: ('chatbot',)
--------------------------------------------------------------------------------
Messages: 2 Next: ('tools',)
--------------------------------------------------------------------------------
Messages: 1 Next: ('chatbot',)
--------------------------------------------------------------------------------
Messages: 0 Next: ('__start__',)
--------------------------------------------------------------------------------
StateSnapshot(values={'messages': [HumanMessage(content='랭그래프가 언제 출시되었는지 찾아줄래요? 결과를 찾으면 human_assistance 도구를 사용해서 사람에게 확인해줘.',

In [25]:
print(to_replay.next)
print(to_replay.config)

('chatbot',)
{'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f163011-3368-6e62-8002-91fb4c16cbb3'}}


In [14]:
# 저장된 시점으로 되감아 실행합니다
for event in graph.stream(None, to_replay.config, stream_mode="values"):
    if "messages" in event:
        event["messages"][-1].pretty_print()

================================= Tool Message =================================
Name: tavily_search

{"query": "랭그래프 출시일", "response_time": 0.55, "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.facebook.com/groups/langchainkr/posts/3589869351148771/", "title": "LangChain과 LangGraph의 1.0 소식입니다   - Facebook", "content": "... 랭그래프. [발표자료] AI 에이전트 101. AIFACTORY.SPACE. [발표자료] AI 에이전트 101. 세미나 소개 일시: 2026년 1월 6일 (화) 오후 8시~9", "score": 0.019719128, "raw_content": null}, {"url": "https://www.youtube.com/watch?v=G8jrAA2bPnA", "title": "LangChain 이 만든 #LangGraph 출시! LangGraph 의 멀티 에이전트 ...", "content": "=sharing 랭체인 튜토리얼 무료 전자책(wikidocs) https://wikidocs.net/book/14314 ✓ 랭 ... Graph Theory 만 생각하세요. Catch Up AI•2.2K", "score": 0.011331754, "raw_content": null}], "request_id": "94b0fc3d-f28a-40ae-9bb6-daed35c0607b"}
================================== Ai Message ==================================
Tool Calls:
  human_assistance (call_s9fdvVEb5I2wupCDnwRs